# Lab - Unit 1: the starter on the real simulator

racekit says the starter takes a median of 870.50 s on `monza-like`, flat sector after flat sector,
because it holds 20 m/s everywhere.
racekit is a model.
**What does the same file do on the real race: CARLA 0.9.12, the Monza map, the official runner?**

In this lab you run `agents/unit-01.py` on the official simulator,
with a small recorder added to its `step()` that writes one telemetry row per tick
in the same v1 CSV format racekit uses.
Then this notebook analyses your recording with exactly the code the unit taught:
`read_csv`, `sector_splits` and the plots.

How to use it:

- The cells tagged `no-exec` are for **your machine** (a Windows PC that can run CARLA).
  Run the shell lines in an Anaconda Prompt, one at a time; they are shown with `!` only so the notebook stays
  valid.
- The analysis cells run anywhere. In CI they run against a racekit-made sample recording,
  so the book always proves they work; on your machine they run against your own recording.

> **Never commit your results.**
> This repository is public.
> Your recordings, screenshots and logs stay on your own machine:
> write them **outside the repository** (the setup below uses `%USERPROFILE%\roar-runs\`).
> The repository ignores data files under `labs/`, and a CI test fails if anything other than a notebook
> or a README is ever tracked there, but the first line of defence is not putting them here at all.

In [ ]:
import os

TELEMETRY_PATH = os.environ.get("ROAR_LAB_CSV", "telemetry.csv")

The parameters cell above is the **only** place a recording's file name appears.
Set the environment variable `ROAR_LAB_CSV` to your recording's full path
(outside the repository) before you start Jupyter, and both the recorder and this notebook use it.

## 1. Set up the official race (once)

The official environment, pinned (design 013 §1.1, `roar/reference/upstream.md`):

| Piece | Version |
|---|---|
| CARLA server | 0.9.12, with the series' **Monza map v1.1** package (a Windows AMD64 executable) |
| Python | **3.8**, in a conda environment; NumPy below 1.25 |
| `ROAR_PY` | `augcog/ROAR_PY` at commit `3cbb087ccc5198248815b57742a945e81ef55b64` |
| `ROAR_Competition` | `augcog/ROAR_Competition` at commit `ab19e1b` |

Download the Monza v1.1 package from the link in the series' instructions and unzip it.
Then set up Python.
Follow each repository's own README at the pinned commit where it differs from this outline:
the book describes the steps and never copies upstream code.

In [ ]:
# Anaconda Prompt, on the Windows machine. Not run in CI.
# 1. the Python 3.8 environment the official runner expects
!conda create -n roar_competition python=3.8 -y
!conda activate roar_competition
!pip install "numpy<1.25" carla==0.9.12
# 2. ROAR_PY (roar_py_interface, roar_py_carla) at the pinned commit
!git clone https://github.com/augcog/ROAR_PY
!git -C ROAR_PY checkout 3cbb087ccc5198248815b57742a945e81ef55b64
# the repository root has no setup.py: install its requirements, then its two packages
!pip install -r ROAR_PY\requirements.txt
!pip install -e ROAR_PY\roar_py_core -e ROAR_PY\roar_py_carla
# 3. the competition runner at the pinned commit
!git clone https://github.com/augcog/ROAR_Competition
!git -C ROAR_Competition checkout ab19e1b
# 4. your submission: the book's starter, copied over the competition's submission.py
!copy %PY4KIDS%\roar\projects\racekit\agents\unit-01.py ROAR_Competition\competition_code\submission.py

`%PY4KIDS%` stands for your clone of this repository.
The copy is the same file racekit raced in the unit; the next section adds the recorder to it.

## 2. The recorder

The official runner does not hand you telemetry, so the submission records its own.
Two things are harder than they look.

**Progress.** The race's progress is cumulative over 3 laps, but the starter only knows its index
into the one-lap waypoint list.
`racekit.telemetry.CumulativeProgress` turns that one-lap index into cumulative progress:
a lap wrap (the index going from at least 0.9·n to below 0.1·n) adds n,
and a **respawn** resets it to 0.
The respawn is detected from the explicit signal, the collision sensor's impulse above 100,
the official runner's major-collision threshold, which the agent sees on its first step after the hold.
Position alone cannot tell a crash near the start line from driving on.

**The clock.** The agent has no race clock; it counts its own steps.
The runner holds a respawned car for 20 ticks **without calling `step()`**,
so on a respawn the recorder writes those 20 ticks itself: one row per hold tick, with `held` = 1 and
progress 0, at the same times as the `held` rows of racekit's own telemetry.
The last hold row and the first row after it share a time: the hold ends at the instant `step()` is
called again.
The hold rows are the respawn signal **in the file**.
A crash before the car passes its first waypoint leaves progress at 0 before and after the crash,
so the progress column alone would never show the restart, and `sector_splits` would count the
lost time and the hold in sector 1.

**A stale first step.** On CARLA the official runner does not read the sensors again after the
respawn, so the first `step()` after the hold sees the impulse together with the **crash pose**:
the location, rotation and velocity from before the respawn. The start pose arrives one step later.
racekit already shows the start pose on that step; CARLA does not.

Here is `CumulativeProgress` exactly as racekit ships it.
It is plain Python with no imports, so it runs on 3.8 and can be pasted into a submission.

In [ ]:
from inspect import getsource

from racekit.telemetry import CumulativeProgress

print(getsource(CumulativeProgress))

Paste the block below into `submission.py`:

- the two lines of this notebook's **parameters cell** go at the top of the file (they define `TELEMETRY_PATH`);
- the constants and the `CumulativeProgress` class go after the imports;
- the three methods go inside `RoarCompetitionSolution`.

The recorder only **writes**: `"w"` once, for the header, and `"a"` for every row,
so a crash mid-race still leaves every row recorded so far on disk.
The header is the v1 meta line that `read_csv` validates, with `n = len(maneuverable_waypoints)`.

In [ ]:
# lab-recorder: the block to paste into submission.py (the steps above say where each part goes)
# --- after the imports of submission.py (Python 3.8-safe) ---
LAPS = 3
SECTORS_PER_LAP = 3
CONTROL_DT = 0.05  # s per control tick
RESPAWN_HOLD_TICKS = 20  # ticks the runner holds a respawned car without calling step()


class CumulativeProgress:  # pasted from the printout above, docstring trimmed
    RESPAWN_IMPULSE = 100.0  # N·s: the official runner's major-collision threshold
    WRAP_HIGH = 0.9
    WRAP_LOW = 0.1

    def __init__(self, n: int) -> None:
        if n < 1:
            raise ValueError("n (waypoints per lap) must be at least 1")
        self.n = n
        self.offset = 0
        self.last_index = None
        self.respawned = False

    def update(self, lap_index: int, collision_impulse) -> int:
        impulse = sum(float(c) * float(c) for c in collision_impulse) ** 0.5
        self.respawned = impulse > self.RESPAWN_IMPULSE
        if self.respawned:
            self.offset = 0
        elif (self.last_index is not None and self.last_index >= self.WRAP_HIGH * self.n
              and lap_index < self.WRAP_LOW * self.n):
            self.offset += self.n
        self.last_index = lap_index
        return self.offset + lap_index


# --- inside the existing class: add these three methods to it ---
class RoarCompetitionSolution:
    def start_recording(self) -> None:
        """Call last in initialize(): the race starts at self.current_index."""
        n = len(self.maneuverable_waypoints)
        self.start_index = self.current_index
        self.progress_tracker = CumulativeProgress(n)
        self.tick = 0
        meta = f"progress_total={LAPS * n} laps={LAPS} per_lap={SECTORS_PER_LAP} waypoints_per_lap={n}"
        with open(TELEMETRY_PATH, "w") as handle:
            handle.write(f"# racekit-telemetry v1 {meta}\n")
            handle.write("t,x,y,yaw,v,throttle,steer,brake,progress,offset,held\n")

    def reanchor_after_respawn(self) -> None:
        """Call in step() right after advance_index(): a respawn restarts the race at the start."""
        impulse = self.collision_sensor.get_last_gym_observation()
        if sum(float(c) * float(c) for c in impulse) ** 0.5 > CumulativeProgress.RESPAWN_IMPULSE:
            # Not nearest_index(location): on CARLA this step's location is still the crash pose.
            self.current_index = self.start_index

    def record(self, location, yaw: float, speed: float, throttle: float, steer: float, brake: float) -> None:
        """Call in step() just before apply_action(): one row per tick."""
        n = len(self.maneuverable_waypoints)
        lap_index = (self.current_index - self.start_index) % n
        impulse = self.collision_sensor.get_last_gym_observation()
        progress = self.progress_tracker.update(lap_index, impulse)
        x, y = float(location[0]), float(location[1])
        with open(TELEMETRY_PATH, "a") as handle:
            if self.progress_tracker.respawned:
                # the hold ticks step() was not called for: held = 1, progress 0, controls 0
                for _ in range(RESPAWN_HOLD_TICKS):
                    self.tick += 1
                    handle.write(f"{self.tick * CONTROL_DT:.6f},{x:.6f},{y:.6f},{yaw:.6f},0.000000,"
                                 "0.000000,0.000000,0.000000,0,0.000000,1\n")
            row = (self.tick * CONTROL_DT, x, y, yaw, speed, throttle, steer, brake)
            # offset is 0: the real runner does not report it to the agent
            handle.write(",".join(f"{value:.6f}" for value in row) + f",{progress},0.000000,0\n")
        self.tick += 1

Then wire the three methods into the starter's lifecycle (three new lines):

```python
    async def initialize(self) -> None:
        location = self.location_sensor.get_last_gym_observation()
        self.current_index = self.nearest_index(location)
        self.start_recording()                                          # new

    async def step(self) -> None:
        location = ...; yaw = ...; speed = ...                          # as before
        self.current_index = self.advance_index(location)              # as before
        self.reanchor_after_respawn()                                   # new
        ...                                                             # steer, throttle, brake as before
        self.record(location, yaw, speed, throttle, steer, brake)       # new
        await self.vehicle.apply_action({...})
```

Without `reanchor_after_respawn`, the starter would keep its index from before the crash, and the
recorder would see a one-lap index far from 0 right after the restart.
It sets the index back to `start_index`, the one `initialize()` found, and it reads no location to do
it: on CARLA that step's location is still the crash pose, so `nearest_index(location)`, or the forward
search in `advance_index`, would find the crash point again
(and a crash in the last tenth of a lap would then look like a lap wrap one step later).
That is also why it runs **after** `advance_index`.
In a CARLA recording the 20 hold rows and the first row after them all carry the crash pose's
`x`, `y` and `yaw` (the recorder writes the hold rows from that stale step), and that first row also carries
the crash pose's `v`; the next row is at the start.
racekit's own hold rows sit at the start pose, so a crash looks different in the two plots,
but the times and the `held` column agree, and `sector_splits` treats both files the same way.

Before you race on CARLA, race the edited file on racekit (`run_race`) and run `check_submission` on it:
a recorder that raises in `step()` costs you a CARLA session, not a racekit second.
Writing one row per tick takes well under a millisecond; `check_submission` reports the step-time p99.

## 3. Race

1. Start the Monza v1.1 CARLA server (the executable from the unzipped package) and wait for the map to load.
2. In the Anaconda Prompt, set `ROAR_LAB_CSV` to a path **outside the repository**,
   for example `set ROAR_LAB_CSV=%USERPROFILE%\roar-runs\monza-run-1.csv`.
3. From `ROAR_Competition\competition_code`, run the official runner (below).
   Note the time and the collision count it reports: that is the score; your recording explains it.
4. Repeat for at least 3 runs, each to its own file: CARLA is not repeatable,
   so one run lies here just as it did in racekit.

In [ ]:
# Anaconda Prompt, from ROAR_Competition\competition_code, with the CARLA server running. Not run in CI.
!python competition_runner.py

## 4. Analyse the recording

Start Jupyter with `ROAR_LAB_CSV` pointing at one recording and run from here down.
Set `RACE_FINISHED` to what the runner reported:
a recorder never sees the finishing tick (the race ends before the next `step()`),
so `sector_splits(..., finished=True)` closes the last sector one tick after the last row,
which is only right for a race that really finished.

In [ ]:
from math import isnan

import matplotlib.pyplot as plt
import numpy as np
from racekit.plots import plot_speed_map, plot_telemetry
from racekit.telemetry import read_csv, sector_splits

RACE_FINISHED = True  # set False if the runner reported that the race did not finish

telemetry, meta = read_csv(TELEMETRY_PATH)
n = meta["waypoints_per_lap"]
print(meta)
print(len(telemetry), "rows, from t =", telemetry["t"][0], "to", telemetry["t"][-1], "s")

In [ ]:
splits = sector_splits(telemetry, meta["progress_total"], laps=meta["laps"], per_lap=meta["per_lap"],
                       finished=RACE_FINISHED)

# a respawn in a recording: a run of held rows (the recorder writes the 1.0 s hold);
# in a file without held rows, cumulative progress dropping to below 0.1·n
held = telemetry["held"]
progress = telemetry["progress"]
held_rows = np.nonzero(held)[0]
drop_rows = np.nonzero((np.diff(progress) < 0) & (progress[1:] < 0.1 * n))[0] + 1
if len(held_rows):
    respawns = len([row for row in held_rows if row == 0 or not held[row - 1]])
    restart = float(telemetry["t"][held_rows[-1]])  # the race restarts when the last hold ends
else:
    respawns = len(drop_rows)
    restart = float(telemetry["t"][drop_rows[-1]]) if len(drop_rows) else 0.0
complete = not any(isnan(split) for lap in splits for split in lap)  # an unreached sector is NaN

for lap, row in enumerate(splits, start=1):
    print(f"lap {lap}:", *[f"{split:6.2f}" for split in row])
print("respawns:", respawns)
if respawns:
    print(f"the restarted race starts at t = {restart:.2f} s")
if complete:
    race_time = restart + sum(sum(lap) for lap in splits)
    print(f"race time: {race_time:.2f} s")
speed = telemetry["v"]
driving = (telemetry["t"] > 10.0) & (held == 0)  # a hold row's speed is 0 by construction
print(f"mean speed {np.mean(speed):.2f} m/s, min after the first 10 s "
      f"{min(speed[driving]):.2f} m/s (held rows excluded)")

The speed trace and the driven path, coloured by speed.
The path is drawn without a track underneath: racekit's tracks are not the CARLA map.

In [ ]:
fig, (ax_speed, ax_map) = plt.subplots(1, 2, figsize=(14, 4.5))
plot_telemetry(telemetry, columns=("v",), ax=ax_speed)
ax_speed.set_title("speed (m/s)")
plot_speed_map(telemetry, ax=ax_map)
ax_map.set_title("driven path")
fig.tight_layout()

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(range(1, 10), [split for lap in splits for split in lap], marker="o")
ax.set_xlabel("sector (laps 1-3, 3 sectors each)")
ax.set_ylabel("split (s)")
ax.set_title("sector splits")
ax.grid(True, alpha=0.3)

## 5. Compare with racekit

The same file, raced on racekit's `monza-like` with the unit's jitter (seed 0), side by side with your recording.
`monza-like` is *inspired by* Monza and not the CARLA map,
so compare **shapes**, not seconds: is the mean speed the 20 m/s cap?
Are the splits flat, with only the standing start slower?
Did the car finish clean?

In CI the "recording" is itself a racekit race (the sample), so the two columns agree;
on your machine the left column is CARLA.

In [ ]:
from pathlib import Path

from racekit.cars import KinematicCar
from racekit.runner import run_race

AGENT = Path("..") / ".." / ".." / "projects" / "racekit" / "agents" / "unit-01.py"
model = run_race(AGENT, "monza-like", car=KinematicCar(jitter=0.05), seed=0)
model_splits = sector_splits(model.telemetry, model.progress_total, laps=3, per_lap=3)


def flatness(table):
    """max - min of the splits after lap 1 sector 1 (the standing start)."""
    rest = [split for lap in table for split in lap][1:]
    return max(rest) - min(rest)


print(f"{'':28}{'recording':>12}{'racekit':>12}")
print(f"{'mean speed (m/s)':28}{np.mean(speed):12.2f}{np.mean(model.telemetry['v']):12.2f}")
print(f"{'lap 1 sector 1 (s)':28}{splits[0][0]:12.2f}{model_splits[0][0]:12.2f}")
print(f"{'other splits, max - min (s)':28}{flatness(splits):12.2f}{flatness(model_splits):12.2f}")
print(f"{'respawns':28}{respawns:12d}{len(model.respawn_times):12d}")

WINNER = 319.65  # Summer 2026 series winner, CARLA Monza
REASONABLE = 329.0  # the book's bar: every run clean, median at most this
if complete:
    print(f"\nrace time {race_time:.2f} s: the bar is a clean median <= {REASONABLE:.0f} s "
          f"(winner {WINNER} s), {race_time / REASONABLE:.1f}x away")

## What to take away

- Write down, from at least 3 CARLA runs: each race time, the median and the range, and any collisions.
  The series' run-to-run noise is about 0.3 s; is yours?
- If the CARLA splits are flat and the mean speed sits at the cap, racekit's diagnosis carries over:
  the starter loses time everywhere, uniformly, because it never goes faster than 20 m/s.
  If they are not flat, find the sector where they differ; that is the first place racekit and CARLA disagree.
- On racekit the clean starter is about 2.6 times slower than the reasonable-ranking bar;
  the last cell printed your CARLA ratio.
  The rest of the book closes that gap, one measured unit at a time.

> **One last check: never commit your results.**
> This repository is public.
> Your recordings, screenshots and logs stay on your own machine, in the folder outside the repository
> that `ROAR_LAB_CSV` pointed at (Section 3 used `%USERPROFILE%\roar-runs\`).
> Run `git status` in your clone before you commit anything: none of them should be listed.